In [1]:
!pip install -q transformers accelerate

In [2]:
import torch
import inspect

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)

In [3]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

MAX_NEW_TOKENS = 256

DEVICE = "cuda"
DTYPE = torch.bfloat16

In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
).to(DEVICE)

model.eval()

print("Model:", type(model).__name__)
print("Device:", model.device)
print("Dtype:", model.dtype)

print("Layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("KV heads:", model.config.num_key_value_heads)
print("Hidden size:", model.config.hidden_size)

head_dim = (
    model.config.hidden_size
    // model.config.num_attention_heads
)

print("Head dimension:", head_dim)

print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model: Qwen2ForCausalLM
Device: cuda:0
Dtype: torch.bfloat16
Layers: 24
Attention heads: 14
KV heads: 2
Hidden size: 896
Head dimension: 64
EOS token: '<|im_end|>'
EOS token ID: 151645


In [5]:
def get_kv_cache_bytes(past_key_values):
    key_bytes = 0
    value_bytes = 0

    for layer in past_key_values.layers:
        key_bytes += (
            layer.keys.numel()
            * layer.keys.element_size()
        )

        value_bytes += (
            layer.values.numel()
            * layer.values.element_size()
        )

    return key_bytes, value_bytes


def bytes_to_mib(num_bytes):
    return num_bytes / (1024 ** 2)


def print_cache_stats(past_key_values):
    key_bytes, value_bytes = get_kv_cache_bytes(
        past_key_values
    )

    total_bytes = key_bytes + value_bytes

    print(
        "KV tokens:",
        past_key_values.get_seq_length()
    )

    print(
        f"Key cache:   {bytes_to_mib(key_bytes):.4f} MiB"
    )

    print(
        f"Value cache: {bytes_to_mib(value_bytes):.4f} MiB"
    )

    print(
        f"Total KV:    {bytes_to_mib(total_bytes):.4f} MiB"
    )

In [6]:
def timed_cuda_call(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    torch.cuda.synchronize()

    start.record()

    result = fn()

    end.record()

    torch.cuda.synchronize()

    elapsed_ms = start.elapsed_time(end)

    return result, elapsed_ms

In [7]:
def generate_from_cache(
    outputs,
    past_key_values,
    generated_ids,
    attention_mask,
    max_new_tokens=MAX_NEW_TOKENS,
):

    assert (
        generated_ids.shape[1]
        == past_key_values.get_seq_length()
    )

    assert (
        attention_mask.shape[1]
        == past_key_values.get_seq_length()
    )

    next_token = torch.argmax(
        outputs.logits[:, -1, :],
        dim=-1,
        keepdim=True,
    )

    generated_token_ids = []
    decode_times_ms = []

    finished = False

    for _ in range(max_new_tokens):

        # --------------------------------
        # 1. Add predicted token
        # --------------------------------

        generated_ids = torch.cat(
            [generated_ids, next_token],
            dim=-1,
        )

        generated_token_ids.append(
            next_token.item()
        )

        # --------------------------------
        # 2. Extend attention mask by one
        # --------------------------------

        attention_mask = torch.cat(
            [
                attention_mask,
                torch.ones(
                    (attention_mask.shape[0], 1),
                    dtype=attention_mask.dtype,
                    device=attention_mask.device,
                ),
            ],
            dim=-1,
        )

        # --------------------------------
        # 3. Process new token into cache
        # --------------------------------

        def decode_step():
            with torch.no_grad():
                return model(
                    input_ids=next_token,
                    attention_mask=attention_mask,
                    past_key_values=past_key_values,
                    use_cache=True,
                )

        outputs, step_ms = timed_cuda_call(
            decode_step
        )

        decode_times_ms.append(step_ms)

        past_key_values = outputs.past_key_values

        # --------------------------------
        # 4. Correctness invariants
        # --------------------------------

        assert (
            generated_ids.shape[1]
            == past_key_values.get_seq_length()
        )

        assert (
            attention_mask.shape[1]
            == past_key_values.get_seq_length()
        )

        # --------------------------------
        # 5. Natural turn termination
        # --------------------------------

        if (
            next_token.item()
            == tokenizer.eos_token_id
        ):
            finished = True
            break

        # --------------------------------
        # 6. Predict following token
        # --------------------------------

        next_token = torch.argmax(
            outputs.logits[:, -1, :],
            dim=-1,
            keepdim=True,
        )

    return {
        "generated_ids": generated_ids,
        "past_key_values": past_key_values,
        "attention_mask": attention_mask,
        "outputs": outputs,
        "generated_token_ids": generated_token_ids,
        "decode_times_ms": decode_times_ms,
        "finished": finished,
    }

In [8]:
t1_messages = [
    {
        "role": "user",
        "content": (
            "I'm planning a tea party today. "
            "The theme is lavender and the special dessert "
            "is lemon cake. Remember these details for me."
        ),
    }
]

t1_text = tokenizer.apply_chat_template(
    t1_messages,
    tokenize=False,
    add_generation_prompt=True,
)

t1_inputs = tokenizer(
    t1_text,
    return_tensors="pt",
).to(model.device)

generated_ids = (
    t1_inputs["input_ids"].clone()
)

attention_mask = (
    t1_inputs["attention_mask"].clone()
)

print("TURN 1")
print()
print(t1_text)

print(
    "\nPrompt tokens:",
    generated_ids.shape[1]
)

TURN 1

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm planning a tea party today. The theme is lavender and the special dessert is lemon cake. Remember these details for me.<|im_end|>
<|im_start|>assistant


Prompt tokens: 55


In [9]:
def run_t1_prefill():

    with torch.no_grad():

        return model(
            input_ids=t1_inputs["input_ids"],
            attention_mask=attention_mask,
            use_cache=True,
        )


outputs, t1_prefill_ms = timed_cuda_call(
    run_t1_prefill
)

past_key_values = outputs.past_key_values


assert (
    generated_ids.shape[1]
    == past_key_values.get_seq_length()
)

assert (
    attention_mask.shape[1]
    == past_key_values.get_seq_length()
)


print(
    f"Turn 1 prefill: "
    f"{t1_prefill_ms:.3f} ms"
)

print()

print_cache_stats(
    past_key_values
)

Turn 1 prefill: 927.667 ms

KV tokens: 55
Key cache:   0.3223 MiB
Value cache: 0.3223 MiB
Total KV:    0.6445 MiB


In [10]:
t1_result = generate_from_cache(
    outputs=outputs,
    past_key_values=past_key_values,
    generated_ids=generated_ids,
    attention_mask=attention_mask,
)

generated_ids = (
    t1_result["generated_ids"]
)

past_key_values = (
    t1_result["past_key_values"]
)

attention_mask = (
    t1_result["attention_mask"]
)

outputs = (
    t1_result["outputs"]
)

t1_generated = (
    t1_result["generated_token_ids"]
)

t1_decode_times = (
    t1_result["decode_times_ms"]
)

t1_finished = (
    t1_result["finished"]
)


print(
    "Naturally terminated:",
    t1_finished
)

print(
    "Generated tokens:",
    len(t1_generated)
)

print(
    f"Total decode time: "
    f"{sum(t1_decode_times):.3f} ms"
)

print(
    f"Mean decode/token: "
    f"{sum(t1_decode_times) / len(t1_decode_times):.3f} ms"
)

print()

print_cache_stats(
    past_key_values
)

print("\nConversation:")

print(
    tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=False,
    )
)


assert t1_finished, (
    "Turn 1 hit MAX_NEW_TOKENS before EOS. "
    "Do not continue to Turn 2."
)

Naturally terminated: True
Generated tokens: 123
Total decode time: 3562.044 ms
Mean decode/token: 28.960 ms

KV tokens: 178
Key cache:   1.0430 MiB
Value cache: 1.0430 MiB
Total KV:    2.0859 MiB

Conversation:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm planning a tea party today. The theme is lavender and the special dessert is lemon cake. Remember these details for me.<|im_end|>
<|im_start|>assistant
Great! Here are the details for your tea party and lemon cake:

**Tea Party Details:**
- Date: [Insert Date]
- Time: [Insert Time]
- Location: [Insert Location]
- Guests: [Insert Guests]
- Theme: Lavender and Lemon
- Dessert: Lemon cake

**Lemon Cake Details:**
- Ingredients: [List of ingredients]
- Preparation: [Instructions for making the lemon cake]
- Serving: [Instructions for serving the lemon cake]

Please let me know if you need any additional details or if you have any specific requests for the party.<|i

In [11]:
t2_user_text = (
    "What theme and special dessert did I choose?"
)

t2_suffix = (
    "<|im_start|>user\n"
    + t2_user_text
    + "<|im_end|>\n"
    + "<|im_start|>assistant\n"
)

t2_inputs = tokenizer(
    t2_suffix,
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)


t2_start = (
    past_key_values.get_seq_length()
)

t2_input_tokens = (
    t2_inputs["input_ids"].shape[1]
)


print("TURN 2 SUFFIX")
print()

print(
    tokenizer.decode(
        t2_inputs["input_ids"][0],
        skip_special_tokens=False,
    )
)

print()

print(
    "Existing cached tokens:",
    t2_start
)

print(
    "New Turn 2 tokens:",
    t2_input_tokens
)

TURN 2 SUFFIX

<|im_start|>user
What theme and special dessert did I choose?<|im_end|>
<|im_start|>assistant


Existing cached tokens: 178
New Turn 2 tokens: 17


In [12]:
new_attention = torch.ones(
    (
        attention_mask.shape[0],
        t2_input_tokens,
    ),
    dtype=attention_mask.dtype,
    device=attention_mask.device,
)

attention_mask = torch.cat(
    [
        attention_mask,
        new_attention,
    ],
    dim=-1,
)


print(
    "Past tokens:",
    t2_start
)

print(
    "New tokens:",
    t2_input_tokens
)

print(
    "Attention mask length:",
    attention_mask.shape[1]
)

assert (
    attention_mask.shape[1]
    == t2_start + t2_input_tokens
)

Past tokens: 178
New tokens: 17
Attention mask length: 195


In [13]:
generated_ids = torch.cat(
    [
        generated_ids,
        t2_inputs["input_ids"],
    ],
    dim=-1,
)


def run_t2_prefill():

    with torch.no_grad():

        return model(
            input_ids=t2_inputs["input_ids"],
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )


outputs, t2_prefill_ms = timed_cuda_call(
    run_t2_prefill
)

past_key_values = (
    outputs.past_key_values
)

t2_prefill_end = (
    past_key_values.get_seq_length()
)


assert (
    t2_prefill_end
    == t2_start + t2_input_tokens
)

assert (
    generated_ids.shape[1]
    == past_key_values.get_seq_length()
)

assert (
    attention_mask.shape[1]
    == past_key_values.get_seq_length()
)


print(
    f"Turn 2 incremental prefill: "
    f"{t2_prefill_ms:.3f} ms"
)

print(
    "Cache before:",
    t2_start
)

print(
    "Tokens appended:",
    t2_input_tokens
)

print(
    "Cache after:",
    t2_prefill_end
)

print()

print_cache_stats(
    past_key_values
)

Turn 2 incremental prefill: 204.120 ms
Cache before: 178
Tokens appended: 17
Cache after: 195

KV tokens: 195
Key cache:   1.1426 MiB
Value cache: 1.1426 MiB
Total KV:    2.2852 MiB


In [14]:
t2_result = generate_from_cache(
    outputs=outputs,
    past_key_values=past_key_values,
    generated_ids=generated_ids,
    attention_mask=attention_mask,
)

generated_ids = (
    t2_result["generated_ids"]
)

past_key_values = (
    t2_result["past_key_values"]
)

attention_mask = (
    t2_result["attention_mask"]
)

outputs = (
    t2_result["outputs"]
)

t2_generated = (
    t2_result["generated_token_ids"]
)

t2_decode_times = (
    t2_result["decode_times_ms"]
)

t2_finished = (
    t2_result["finished"]
)


print(
    "Naturally terminated:",
    t2_finished
)

print(
    "Generated tokens:",
    len(t2_generated)
)

print(
    f"Total decode time: "
    f"{sum(t2_decode_times):.3f} ms"
)

print(
    f"Mean decode/token: "
    f"{sum(t2_decode_times) / len(t2_decode_times):.3f} ms"
)

print()

print_cache_stats(
    past_key_values
)

print("\nTurn 2 response:")

print(
    tokenizer.decode(
        t2_generated,
        skip_special_tokens=False,
    )
)

print("\nFull conversation:")

print(
    tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=False,
    )
)

Naturally terminated: True
Generated tokens: 19
Total decode time: 516.220 ms
Mean decode/token: 27.169 ms

KV tokens: 214
Key cache:   1.2539 MiB
Value cache: 1.2539 MiB
Total KV:    2.5078 MiB

Turn 2 response:
I chose the theme of lavender and lemon, and the special dessert was a lemon cake.<|im_end|>

Full conversation:
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
I'm planning a tea party today. The theme is lavender and the special dessert is lemon cake. Remember these details for me.<|im_end|>
<|im_start|>assistant
Great! Here are the details for your tea party and lemon cake:

**Tea Party Details:**
- Date: [Insert Date]
- Time: [Insert Time]
- Location: [Insert Location]
- Guests: [Insert Guests]
- Theme: Lavender and Lemon
- Dessert: Lemon cake

**Lemon Cake Details:**
- Ingredients: [List of ingredients]
- Preparation: [Instructions for making the lemon cake]
- Serving: [Instructions for serving the lemon c

In [15]:
print(
    "Token sequence length:",
    generated_ids.shape[1]
)

print(
    "Attention mask length:",
    attention_mask.shape[1]
)

print(
    "KV cache length:",
    past_key_values.get_seq_length()
)


assert (
    generated_ids.shape[1]
    == attention_mask.shape[1]
    == past_key_values.get_seq_length()
)

print("\nAll baseline invariants passed.")

Token sequence length: 214
Attention mask length: 214
KV cache length: 214

All baseline invariants passed.


In [16]:
print(
    "Cache type:",
    type(past_key_values)
)

print(
    "Number of layers:",
    len(past_key_values.layers)
)

print(
    "Sequence length:",
    past_key_values.get_seq_length()
)


for layer_idx in [0, 10, 23]:

    layer = past_key_values.layers[
        layer_idx
    ]

    print(
        f"\nLayer {layer_idx}"
    )

    print(
        "K:",
        layer.keys.shape,
        layer.keys.dtype,
        layer.keys.device,
    )

    print(
        "V:",
        layer.values.shape,
        layer.values.dtype,
        layer.values.device,
    )

Cache type: <class 'transformers.cache_utils.DynamicCache'>
Number of layers: 24
Sequence length: 214

Layer 0
K: torch.Size([1, 2, 214, 64]) torch.bfloat16 cuda:0
V: torch.Size([1, 2, 214, 64]) torch.bfloat16 cuda:0

Layer 10
K: torch.Size([1, 2, 214, 64]) torch.bfloat16 cuda:0
V: torch.Size([1, 2, 214, 64]) torch.bfloat16 cuda:0

Layer 23
K: torch.Size([1, 2, 214, 64]) torch.bfloat16 cuda:0
V: torch.Size([1, 2, 214, 64]) torch.bfloat16 cuda:0


In [17]:
token_ids = generated_ids[0]

print(
    f"{'POS':>4} | "
    f"{'TOKEN ID':>8} | "
    f"TOKEN"
)

print("-" * 55)


for pos, token_id in enumerate(token_ids):

    token_text = tokenizer.decode(
        [token_id.item()],
        skip_special_tokens=False,
    )

    print(
        f"{pos:4d} | "
        f"{token_id.item():8d} | "
        f"{repr(token_text)}"
    )

 POS | TOKEN ID | TOKEN
-------------------------------------------------------
   0 |   151644 | '<|im_start|>'
   1 |     8948 | 'system'
   2 |      198 | '\n'
   3 |     2610 | 'You'
   4 |      525 | ' are'
   5 |     1207 | ' Q'
   6 |    16948 | 'wen'
   7 |       11 | ','
   8 |     3465 | ' created'
   9 |      553 | ' by'
  10 |    54364 | ' Alibaba'
  11 |    14817 | ' Cloud'
  12 |       13 | '.'
  13 |     1446 | ' You'
  14 |      525 | ' are'
  15 |      264 | ' a'
  16 |    10950 | ' helpful'
  17 |    17847 | ' assistant'
  18 |       13 | '.'
  19 |   151645 | '<|im_end|>'
  20 |      198 | '\n'
  21 |   151644 | '<|im_start|>'
  22 |      872 | 'user'
  23 |      198 | '\n'
  24 |       40 | 'I'
  25 |     2776 | "'m"
  26 |     9115 | ' planning'
  27 |      264 | ' a'
  28 |    15243 | ' tea'
  29 |     4614 | ' party'
  30 |     3351 | ' today'
  31 |       13 | '.'
  32 |      576 | ' The'
  33 |     6912 | ' theme'
  34 |      374 | ' is'
  35 |    80360 | ' lav

In [18]:
print(
    "\nTurn 2 begins at position:",
    t2_start
)

print(
    "Turn 2 assistant generation begins at:",
    t2_prefill_end
)

print(
    "Final cache position:",
    past_key_values.get_seq_length() - 1
)


Turn 2 begins at position: 178
Turn 2 assistant generation begins at: 195
Final cache position: 213


In [19]:
key_bytes, value_bytes = (
    get_kv_cache_bytes(
        past_key_values
    )
)

total_kv_bytes = (
    key_bytes + value_bytes
)


print("=" * 50)
print("FULL-CACHE BASELINE")
print("=" * 50)

print(
    f"Turn 1 prompt tokens:      "
    f"{t1_inputs['input_ids'].shape[1]}"
)

print(
    f"Turn 1 generated tokens:   "
    f"{len(t1_generated)}"
)

print(
    f"Turn 1 prefill:            "
    f"{t1_prefill_ms:.3f} ms"
)

print(
    f"Turn 1 decode/token:       "
    f"{sum(t1_decode_times) / len(t1_decode_times):.3f} ms"
)

print()

print(
    f"Turn 2 new input tokens:   "
    f"{t2_input_tokens}"
)

print(
    f"Turn 2 generated tokens:   "
    f"{len(t2_generated)}"
)

print(
    f"Turn 2 prefill:            "
    f"{t2_prefill_ms:.3f} ms"
)

print(
    f"Turn 2 decode/token:       "
    f"{sum(t2_decode_times) / len(t2_decode_times):.3f} ms"
)

print()

print(
    f"Final KV tokens:           "
    f"{past_key_values.get_seq_length()}"
)

print(
    f"Final KV size:             "
    f"{bytes_to_mib(total_kv_bytes):.4f} MiB"
)

print(
    f"Final sequence length:     "
    f"{generated_ids.shape[1]}"
)

print(
    f"Final attention length:    "
    f"{attention_mask.shape[1]}"
)

print("=" * 50)

FULL-CACHE BASELINE
Turn 1 prompt tokens:      55
Turn 1 generated tokens:   123
Turn 1 prefill:            927.667 ms
Turn 1 decode/token:       28.960 ms

Turn 2 new input tokens:   17
Turn 2 generated tokens:   19
Turn 2 prefill:            204.120 ms
Turn 2 decode/token:       27.169 ms

Final KV tokens:           214
Final KV size:             2.5078 MiB
Final sequence length:     214
Final attention length:    214
